# Olympics ML — Model Improvements & Comparisons

## Section 1 — Setup & Data Preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../data/athlete_events.csv')
regions = pd.read_csv('../data/noc_regions.csv')

In [3]:
df = pd.merge(df, regions, on='NOC', how='left')

In [4]:
print(df.shape)
print(df.columns.tolist())
print(df.info())
print(df.describe())

(271116, 17)
['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year', 'Season', 'City', 'Sport', 'Event', 'Medal', 'region', 'notes']
<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 17 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
 15  region  270746 non-null  str    
 16  notes   5039 non-null    str    
dtypes: float64(3), 

In [5]:
df.drop(columns=['notes'], inplace=True)

In [6]:
df['Medal'] = df['Medal'].fillna('No Medal')

In [7]:
print(df['Medal'].value_counts())
print(df['Medal'].isnull().sum())

Medal
No Medal    231333
Gold         13372
Bronze       13295
Silver       13116
Name: count, dtype: int64
0


In [8]:
df['Medal_Won'] = df['Medal'].apply(lambda x: 0 if x == 'No Medal' else 1)

In [9]:
df['Age'] = df.groupby('Sport')['Age'].transform(
    lambda x: x.fillna(x.median())
)

In [10]:
df['Height'] = df.groupby('Sport')['Height'].transform(
    lambda x: x.fillna(x.median())
)

In [11]:
df['Weight'] = df.groupby('Sport')['Weight'].transform(
    lambda x: x.fillna(x.median())
)

In [12]:
df.dropna(subset=['region'], inplace=True)

In [13]:
df['Height'] = df['Height'].fillna(df['Height'].median())
df['Weight'] = df['Weight'].fillna(df['Weight'].median())

In [15]:
print("Shape after cleaning:", df.shape)
print("\nMissing values remaining:")
print(df.isnull().sum())
print("\nMedal_Won value counts:")
print(df['Medal_Won'].value_counts())

Shape after cleaning: (270746, 17)

Missing values remaining:
ID           0
Name         0
Sex          0
Age          0
Height       0
Weight       0
Team         0
NOC          0
Games        0
Year         0
Season       0
City         0
Sport        0
Event        0
Medal        0
region       0
Medal_Won    0
dtype: int64

Medal_Won value counts:
Medal_Won
0    230972
1     39774
Name: count, dtype: int64


In [16]:
cols = ['Sex','Age','Height','Weight','Year','Season','Sport','region','Medal_Won']
present = [c for c in cols if c in df.columns]
new_df = df[present].copy()
print("Selected columns:", present)

Selected columns: ['Sex', 'Age', 'Height', 'Weight', 'Year', 'Season', 'Sport', 'region', 'Medal_Won']


In [17]:
new_df.dropna(inplace=True)

In [18]:
from sklearn.preprocessing import LabelEncoder 

In [19]:
le_sex = LabelEncoder()
new_df['Sex'] = le_sex.fit_transform(new_df['Sex'])

In [20]:
le_season = LabelEncoder()
new_df['Season'] = le_season.fit_transform(new_df['Season'])

In [21]:
new_df = pd.get_dummies(new_df, columns=['Sport', 'region'], drop_first=True)

In [22]:
X = new_df.drop('Medal_Won', axis=1)
y = new_df['Medal_Won']

In [23]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nMedal_Won value counts:")
print(y.value_counts())
print("\nData types in X:")
print(X.dtypes.value_counts())

X shape: (270746, 275)
y shape: (270746,)
X_train shape: (216596, 275)
X_test shape: (54150, 275)

Medal_Won value counts:
Medal_Won
0    230972
1     39774
Name: count, dtype: int64

Data types in X:
bool       269
int64        3
float64      3
Name: count, dtype: int64
